# Day 6 (Part 2): Broadcasting, np.where, np.argmax, Stacking & Splitting

Continuation of Day 6 — this notebook covers the second half of the session: broadcasting rules, `np.where`, `np.argmax`, and stacking/splitting.


## Part 1: Broadcasting

**Rule:** compare shapes from the **rightmost dimension leftward**. Two dimensions are compatible if they're equal, or one of them is `1`. If the arrays have a different number of dimensions, pad the smaller one with `1`s on the *left* first, then compare every pair — not just the last one.

### Example 1: vector + matrix, matching trailing dimension


In [ ]:
import numpy as np

a = np.array([[1, 2, 3],
              [4, 5, 6]])      # shape (2, 3)
b = np.array([10, 20, 30])     # shape (3,)

a + b

array([[11, 22, 33],
       [14, 25, 36]])

> **Key insight:** `b`'s shape `(3,)` matches `a`'s trailing dimension (3), so `b` is applied to every row.

### Example 2: mismatched trailing dimension → error


In [ ]:
c = np.array([1, 2])   # shape (2,)

try:
    a + c
except ValueError as e:
    print("ValueError:", e)

ValueError: operands could not be broadcast together with shapes (2,3) (2,) 


> **Key insight:** trailing dims are `3` vs `2` — neither equal nor `1` — so broadcasting fails immediately. Matching vector length to the *last* axis isn't enough on its own; it has to actually satisfy the rule.

### Example 3: different number of dimensions — pad with 1s, check every pair


In [3]:
x = np.ones((2, 1, 3))   # shape (2, 1, 3)
y = np.ones((4, 3))       # shape (4, 3) -> padded to (1, 4, 3)

result = x + y
result.shape

(2, 4, 3)

> **Key insight:** `y` (2D) gets padded to `(1, 4, 3)` to match `x`'s 3 dimensions. Then check right to left:
> - `3` vs `3` → match
> - `1` vs `4` → compatible, `1` stretches to `4`
> - `2` vs `1` (padded) → compatible, `1` stretches to `2`
>
> Result shape: `(2, 4, 3)`. This is the part that's easy to get wrong — checking only the rightmost pair and stopping there.

## Part 2: `np.where`

`np.where(condition, x, y)` — wherever `condition` is `True`, take from `x`; wherever `False`, take from `y`.


In [4]:
arr = np.array([1, -2, 3, -4, 5])
np.where(arr > 0, arr, 0)

array([1, 0, 3, 0, 5])

> **Key insight:** negatives replaced with `0`, positives kept as-is. Same broadcasting rules apply to `condition`, `x`, and `y`.

## Part 3: `np.argmax`

Returns the **index** of the maximum value, not the value itself.

- **No axis:** the array is flattened first, then a single scalar index is returned.
- **`axis=N`:** the axis you name is the one that gets **collapsed/removed** from the result shape.


In [5]:
g = np.array([[1, 5, 3],
              [9, 2, 6]])   # shape (2, 3)

print("no axis (flattened):", np.argmax(g))         # flatten -> [1,5,3,9,2,6], max=9 at index 3
print("axis=0 (collapse rows):   ", np.argmax(g, axis=0))  # per column, which row wins
print("axis=1 (collapse columns):", np.argmax(g, axis=1))  # per row, which column wins

no axis (flattened): 3
axis=0 (collapse rows):    [1 0 1]
axis=1 (collapse columns): [1 0]


> **Key insight (no-axis case):** flattening `g` gives `[1, 5, 3, 9, 2, 6]`. The max is `9` at flattened position `3` — a single scalar, not a per-row/column list. It's easy to jump straight to the axis=1 answer here by accident.
>
> **Key insight (axis rule):** the axis you specify is the one that disappears from the output shape. `axis=0` collapses rows → one answer per column. `axis=1` collapses columns → one answer per row.
>
> **`keepdims`:** newer NumPy supports `keepdims=True`, which keeps the collapsed axis but sets its size to `1` instead of removing it — same idea as slice-vs-integer indexing keeping a dimension alive.
>
> **Is it a view?** No. `argmax` computes brand-new index values; it isn't selecting/rearranging existing data the way slicing does, so there's no shared-memory view. Aggregate/computation functions (`argmax`, `where`, `sum`, etc.) always return new arrays.

## Part 4: Stacking


In [6]:
r1 = np.array([1, 2, 3])
r2 = np.array([4, 5, 6])

np.vstack([r1, r2])

array([[1, 2, 3],
       [4, 5, 6]])

> **Key insight:** `vstack` stacks the arrays as **rows**, not interleaved columns. Result shape `(2, 3)`.


In [7]:
np.hstack([r1, r2])

array([1, 2, 3, 4, 5, 6])

> **Key insight:** on 1D arrays, `hstack` just concatenates end-to-end into one longer 1D array — there's no second axis yet to stack "horizontally" along. Result shape `(6,)`.

## Part 5: Splitting


In [8]:
m = np.arange(12).reshape(3, 4)

np.split(m, 3, axis=0)

[array([[0, 1, 2, 3]]), array([[4, 5, 6, 7]]), array([[ 8,  9, 10, 11]])]

> **Key insight:** `np.split` works via slicing under the hood, so each piece **keeps its dimensionality** — every piece is shape `(1, 4)`, not `(4,)`. Same rule as "slice keeps the dimension, integer index drops it" from the fancy-indexing notebook.

## Summary

- **Broadcasting:** compare shapes right-to-left; pad missing leading dims with `1`; every pair must match or contain a `1`. Checking only the last dimension is the classic mistake.
- **`np.where`:** elementwise conditional select, broadcasts all three arguments.
- **`np.argmax`:** no axis → flatten to a scalar index. `axis=N` → that axis is removed from the output shape. Never a view — always a new array of indices.
- **`vstack`/`hstack`:** stack as rows vs. concatenate end-to-end (for 1D inputs).
- **`np.split`:** preserves dimensionality via slicing, doesn't collapse axes.
